# Vault EST: matriz de enroll y reenroll

Validación real del 28-10-2026 contra Vault Enterprise 2.2.0: **41 peticiones, 29 emisiones correctas, 12 rechazos, 8 identidades**. Tres equipos con CN/SAN distintos y claves nuevas en cada emisión correcta. El notebook original `2_PKI_API_EST.ipynb` no se ha modificado.

Esta copia permite consultar la evidencia y repetir el laboratorio mediante Bash, curl, OpenSSL 3 y jq. Vault CLI configura los recursos y kubectl recoge auditoría. Las celdas usan `%%bash`; no hay código de aplicación Python ni instalaciones. Ejecutar desde la raíz del repositorio. Los resultados medidos están en el informe y los ficheros de evidencia; no se simulan outputs de ejecución de este notebook.

## 1. Resultado observado

| Autenticación | Enroll de tres equipos | Reenroll | Clientes |
|---|---|---|---|
| Un usuario/password compartido | 3 OK | 3 OK, presentando además el certificado/clave propio de cada equipo | 1; renovar añade 0 |
| Usuario/password distinto por equipo | 3 OK | 3 OK con el certificado/clave propio | 3; renovar añade 0 |
| Certificado/clave compartido para bootstrap, sin Basic | 3 OK con CN distintos | 3 OK, cada equipo presenta el certificado emitido y su clave | 1 en altas + 3 identidades propias al primer reenroll |
| Operaciones posteriores con certificado propio, sin Basic | 3 OK | 3 OK | Mismas 3 identidades propias; añade 0 |
| Renovar el certificado compartido en sí | El compartido renovado permite otra alta | OK manteniendo su Subject/SAN y cambiando clave | Mantiene su identidad |
| Basic y certificado juntos | OK | OK con el certificado propio del equipo | Se selecciona userpass |

En la transición compartido → certificado propio, se conservan la identidad histórica del compartido y las tres propias: **4 en ese conjunto**. No se fusionaron aliases.

**Controles negativos (no son el flujo correcto):** usar el certificado bootstrap para renovar un Subject de equipo distinto se rechaza (400). El flujo correcto presenta el certificado emitido al equipo y su clave. Sin credenciales, password incorrecto y certificado no confiable se rechazan (401). Basic sin certificado en reenroll se rechaza (400). Cambiar Subject o SAN se rechaza (400). Basic incorrecto más certificado válido no hace fallback. Basic correcto más certificado no confiable sí permite enroll: los dos métodos no se comportan como MFA en esta configuración.

In [ ]:
%%bash
set -euo pipefail
EVIDENCE="evidence/est-auth-matrix-20261028T143051Z"
jq '{status,run_id,analyzed_from_saved_requests,requests,successful_issuances,rejected_requests,unique_serials,unique_fingerprints,identities:(.unique_client_ids|length),certificates}' "$EVIDENCE/summary.json"
jq -r '.[]|[.phase,.clients.entity_clients,.delta]|@tsv' "$EVIDENCE/phase-deltas.json"

## 2. Auditoría, aliases y controles offline

La siguiente celda no contacta con Vault ni emite certificados. Comprueba los resultados HTTP exactos, identidad estable en renovaciones, transición a certificados propios y precedencia de autenticadores. Después muestra la correlación petición → autenticación → entidad. `entity-aliases.json` conserva el alias y accessor; al haberse eliminado los mounts, los campos mount_path/mount_type pueden aparecer nulos.

In [ ]:
%%bash
set -euo pipefail
EVIDENCE="${EST_EVIDENCE_DIR:-evidence/est-auth-matrix-20261028T143051Z}"
jq -e -f scripts/verify_est_auth_matrix.jq "$EVIDENCE/operations.json"
jq -r '.[]|[.step,.http,(.entity_id//"-"),(.auth_paths|join(","))]|@tsv' "$EVIDENCE/operations.json"
jq '.[]|{id,aliases}' "$EVIDENCE/entity-aliases.json"

## 3. Exportación, certificados y limpieza observada

En esta ejecución el contador de clientes pasó de 17 a 25 y las ocho filas de exportación coinciden con las ocho identidades de auditoría. El contador de certificados pasó de 41 a 72: 29 emisiones EST + una hoja de autenticación inicial + una CA. Renovar no necesariamente crea otra identidad, pero sí emite otro certificado.

Se comprobaron cadena, Subject y correspondencia con la clave pública del CSR durante la prueba; una verificación independiente posterior comprobó las 29 cadenas y DNS SAN. Los seriales y fingerprints son distintos. Las claves/passwords temporales y los mounts/política/dispositivo de auditoría creados se eliminaron. Permanecen las evidencias públicas/redactadas, auditoría HMAC del servidor, registros Identity residuales y uso histórico. La siguiente celda verifica la evidencia guardada, no el estado actual del servidor.

In [ ]:
%%bash
set -euo pipefail
EVIDENCE="evidence/est-auth-matrix-20261028T143051Z"
jq -e -s --slurpfile ops "$EVIDENCE/operations.json" '([.[].client_id]|unique)==([$ops[0][].client_id|select(.!=null)]|unique)' "$EVIDENCE/activity-export-focused.jsonl"
for category in auth pki audit policy; do
  jq -e '.remaining|length==0' "$EVIDENCE/cleanup-$category.json"
done
test "$(wc -l < "$EVIDENCE/independent-certificate-verification.txt" | tr -d ' ')" = 29
printf 'Evidencia de exportación, certificados y limpieza: OK\n'

## 4. Repetir el laboratorio completo (opcional)

La ejecución carga `/tmp/vault/config.env` sin imprimir secretos, usa namespace root y contexto `workshop-k8s` (sobrescribible con `KUBE_CONTEXT`). Necesita permisos para administrar PKI/auth/policies/audit e inspeccionar Identity y contadores. El token administrativo no se envía a EST. OpenSSL usa `/opt/homebrew/opt/openssl@3/bin/openssl` por defecto, sobrescribible con `OPENSSL_BIN`.

Crea recursos con nombres únicos y los elimina al terminar. No cambia listeners ni rutas EST globales. **Cada ejecución añade uso histórico**; en esta versión se esperan 8 identidades nuevas y 31 certificados. La CA es de 24h, hojas de 1h y tokens batch de 5 minutos. No compartir la clave privada del laboratorio fuera de esta prueba.

La celda está desactivada por defecto para que consultar este notebook no vuelva a emitir certificados. Para ejecutar, cambiar `RUN_EST_MATRIX=false` por `true`, o lanzar en terminal `bash scripts/test_est_auth_matrix.sh`. No se instalan dependencias.

In [ ]:
%%bash
set -euo pipefail
RUN_EST_MATRIX=false
if [[ "$RUN_EST_MATRIX" == true ]]; then
  bash scripts/test_est_auth_matrix.sh
else
  printf 'Prueba real desactivada. Cambia RUN_EST_MATRIX=true para crear recursos y emitir certificados.\n'
fi

## 5. Límites y reproducción del análisis

Las 41 peticiones se completaron en la primera ejecución. Un error de selección de array en jq interrumpió el resumen y activó la limpieza. Se corrigió y se reconstruyó el resumen con las peticiones/auditoría guardadas, sin reemitir. Por eso `analyzed_from_saved_requests=true`.

Para repetir solo el análisis guardado y refrescar lecturas de Identity/contadores/exportación (sin recrear recursos):

```bash
bash scripts/test_est_auth_matrix.sh --analyze evidence/est-auth-matrix-20261028T143051Z
```

Para validar totalmente offline, utilizar la celda de la sección 2. El análisis con Vault vuelve a consultar contadores globales, por lo que si hay actividad posterior el delta puede incluirla; el inventario de certificados y la auditoría de esta ejecución no cambian.

Los aliases cert observados son CN + accessor del auth mount; **no es una identidad global por CN**. No se han probado otros namespaces/mounts, alias linking, roles cert con certificado leaf fijado, aliases personalizados, revocación/caducidad, proxies, fabricantes ni renovación programada a largo plazo. Los controles Subject/SAN corresponden a estas peticiones sin extensiones de cambio de nombre. Compartir un certificado privado no es una recomendación de seguridad y el contador técnico no sustituye los términos contractuales.

[Informe completo](PKI_EST_SCEP_Validation_Report.md#est-authentication-matrix-enrollment-and-re-enrollment) · [Implementación](scripts/test_est_auth_matrix.sh) · [Vault EST](https://developer.hashicorp.com/vault/docs/secrets/pki/est)

## 6. Repetición aislada del ciclo correcto con certificados

La tabla anterior se ha corregido: el rechazo al usar el bootstrap para renovar otro Subject es un **control negativo**, no el flujo normal. La repetición `20261028T145228Z` utiliza el flujo correcto, sin Basic y con un único autenticador cert:

| Fase | Certificado/clave presentado en TLS | Resultado | Contador de clientes | Incremento |
|---|---|---|---:|---:|
| Antes | — | — | 25 | — |
| Tres altas | Bootstrap compartido | 3 × HTTP 200 | 26 | +1 |
| Primera renovación de los tres equipos | Certificado emitido a cada equipo + su clave | 3 × HTTP 200 | 29 | +3 |
| Segunda renovación de los tres equipos | Certificado ya renovado de cada equipo + su clave | 3 × HTTP 200 | 29 | 0 |

Cada renovación conserva Subject/SAN, presenta exactamente el certificado devuelto en la etapa anterior y solicita una clave nueva. Se comprueban certificado/clave TLS, cadena, serial y fingerprint del certificado usado frente al emitido anteriormente, CSR y certificado nuevo. Las nueve operaciones son correctas: el total histórico del conjunto es una identidad bootstrap más tres identidades propias. No se cambia el mount, rol, política ni namespace entre fases.

El contador de certificados pasó de 72 a 83: nueve hojas EST + bootstrap + CA. La auditoría, Identity y contadores por petición están disponibles; la exportación detallada todavía estaba pendiente al muestrearla. La ejecución finalizó sin errores ni replay del análisis. Recursos temporales y claves eliminados; evidencia pública/redactada e histórico conservados.

Para repetir exactamente este ciclo (crea recursos y añade uso histórico):

```bash
bash scripts/test_est_auth_matrix.sh --cert-lifecycle
```

La siguiente celda solo verifica la evidencia guardada, sin contactar con Vault. [Informe de la repetición](PKI_EST_SCEP_Validation_Report.md#correct-est-certificate-lifecycle-focused-repeat).

In [ ]:
%%bash
set -euo pipefail
EVIDENCE="evidence/est-auth-matrix-20261028T145228Z"
jq -e -f scripts/verify_est_cert_lifecycle.jq "$EVIDENCE/operations.json"
jq '{status,mode,requests,successful_issuances,rejected_requests,analyzed_from_saved_requests,certificates}' "$EVIDENCE/summary.json"
jq -r '.[]|[.phase,.clients.entity_clients,.delta]|@tsv' "$EVIDENCE/phase-deltas.json"
jq -r '.[]|[.step,.tls_certificate,.tls_serial,.serial,.entity_id]|@tsv' "$EVIDENCE/operations.json"
for category in auth pki audit policy; do
  jq -e '.remaining|length==0' "$EVIDENCE/cleanup-$category.json"
done